<a href="https://colab.research.google.com/github/teamogundata-hub/Intelligence-Compliance-classification-Project/blob/main/NFIU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip -q install pdfplumber pandas

In [2]:
import io
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pdfplumber
from google.colab import files

OUTPUT_DIR = Path("/content/icc_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {OUTPUT_DIR}")

Output folder: /content/icc_data


In [3]:
print("Select one or more PDF files from your computer.")

uploaded_files = files.upload()

pdf_files = {
    filename: content
    for filename, content in uploaded_files.items()
    if filename.lower().endswith(".pdf")
}

if not pdf_files:
    raise ValueError("No PDF file was uploaded. Please upload a file ending in .pdf")

print(f"\nPDF files uploaded: {len(pdf_files)}")
for filename in pdf_files:
    print(f"- {filename}")

Select one or more PDF files from your computer.


Saving AdvisoryAndGuidance.pdf to AdvisoryAndGuidance.pdf

PDF files uploaded: 1
- AdvisoryAndGuidance.pdf


In [4]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_pdf_pages(filename: str, pdf_bytes: bytes) -> dict:
    pages = []
    extraction_errors = []

    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            try:
                page_text = clean_text(page.extract_text() or "")

                if page_text:
                    pages.append({
                        "page_number": page_number,
                        "text": page_text,
                        "character_count": len(page_text)
                    })

            except Exception as error:
                extraction_errors.append({
                    "page_number": page_number,
                    "error": str(error)
                })

    full_text = "\n\n".join(page["text"] for page in pages)

    return {
        "document_id": Path(filename).stem.lower().replace(" ", "_"),
        "filename": filename,
        "source_type": "local_pdf_upload",
        "regulator": "NFIU / CBN / Other",
        "document_title": Path(filename).stem,
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
        "full_text": full_text,
        "character_count": len(full_text),
        "pages_extracted": len(pages),
        "page_text": pages,
        "extraction_errors": extraction_errors
    }


documents = []

for filename, pdf_bytes in pdf_files.items():
    print(f"Extracting text: {filename}")

    document = extract_pdf_pages(filename, pdf_bytes)
    documents.append(document)

    print(
        f"  Pages extracted: {document['pages_extracted']} | "
        f"Characters: {document['character_count']:,}"
    )

print(f"\nCompleted extraction for {len(documents)} document(s).")

Extracting text: AdvisoryAndGuidance.pdf
  Pages extracted: 13 | Characters: 19,512

Completed extraction for 1 document(s).


In [5]:
for document in documents:
    print("=" * 100)
    print(f"FILE: {document['filename']}")
    print(f"PAGES EXTRACTED: {document['pages_extracted']}")
    print(f"CHARACTERS: {document['character_count']:,}")
    print("-" * 100)
    print(document["full_text"][:3000])
    print("\n")

FILE: AdvisoryAndGuidance.pdf
PAGES EXTRACTED: 13
CHARACTERS: 19,512
----------------------------------------------------------------------------------------------------
NIGERIAN FINANCIAL INTELLIGENCE UNIT
REF: STR-NFIU-2024-A0001 13 December 2024
GUIDELINES FOR THE IDENTIFICATION, VERIFICATION AND REPORTING OF
SUSPICIOUS TRANSACTIONS RELATED TO MONEY LAUNDERING, FINANCING OF
TERRORISM AND PROLIFERATION OF WEAPONS OF MASS DESTRUCTION
(ML/FT/PF) FOR FINANCIAL INSTITUTIONS
REF: STR-NFIU-2024-A0001 PUBLIC 1

PART ONE
In compliance with its powers under Section 3(1) (a-s) and Section 23 (2) (a)
of the Nigerian Financial Intelligence Unit (Establishment) Act, 2018 and the
Money Laundering (Prevention and Prohibition) Act, 2022 and under its
inherent powers to prevent, mitigate and combat money laundering linked
to cash-based financial dealings, Illicit Financial Flows (IFF) and the
responsibility to protect the integrity of the financial system, this document
is issued for the guidance and

In [6]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

corpus_json_path = OUTPUT_DIR / f"icc_regulatory_corpus_{timestamp}.json"
corpus_csv_path = OUTPUT_DIR / f"icc_regulatory_pages_{timestamp}.csv"

with open(corpus_json_path, "w", encoding="utf-8") as file:
    json.dump(documents, file, ensure_ascii=False, indent=2)

page_records = []

for document in documents:
    for page in document["page_text"]:
        page_records.append({
            "document_id": document["document_id"],
            "filename": document["filename"],
            "document_title": document["document_title"],
            "regulator": document["regulator"],
            "page_number": page["page_number"],
            "text": page["text"],
            "character_count": page["character_count"],
            "label": "",
            "obligation_category": "",
            "control_id": "",
            "review_status": "unreviewed"
        })

pages_df = pd.DataFrame(page_records)
pages_df.to_csv(corpus_csv_path, index=False, encoding="utf-8-sig")

print(f"JSON corpus saved: {corpus_json_path}")
print(f"CSV annotation file saved: {corpus_csv_path}")
print(f"Total extracted pages: {len(pages_df)}")

JSON corpus saved: /content/icc_data/icc_regulatory_corpus_20260826_074442.json
CSV annotation file saved: /content/icc_data/icc_regulatory_pages_20260826_074442.csv
Total extracted pages: 13
